# Lexical analysis

The aim of this part of the analysis is to answer RQ1:

RQ1: Lexical features 
How lexically similar are responses generated by generic and fine-tuned LLMs compared to human therapist responses, based on quantitative measures of lexical richness and complexity?

By lexical richness and complexity, we aim to address the:
- total number of tokens/words
- number of sentences
- average sentence length 
- Type-Token Ratio (TTR) → Unique words / total words

In [1]:
# Install Packages
!pip install spacy
!python -m spacy download en_core_web_sm


Defaulting to user installation because normal site-packages is not writeable
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached numpy-2.3.5-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached pydantic-2.12.4-py3-none-any.whl.metadata (89 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.41.5-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.2/33.2 MB 72.0 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.0/875.0 kB 11.4 MB/s  0:00:00
Using cached pydantic-2.12.4-py3-none-any.whl (463 kB)
Using cached pydantic_core-2.41.5-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 13.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# loading packages
import spacy
#import pandas as pd
import numpy as np

#load spaCy model
nlp = spacy.load("en_core_web_sm")


In [6]:
# Read file 
import pandas as pd

df = pd.read_csv("/work//NLP2025/data/final_df.csv")


In [10]:
import os
os.getcwd()

'/work/NLP2025/src'

In [11]:
# --- word counting function ---
def spacy_word_count(text):
    if not isinstance(text, str) or text.strip() == "":
        return 0
    doc = nlp(text)
    return sum(1 for token in doc if token.is_alpha)


# --- list of columns to compute word counts for ---
cols = ["Context", "Human_response", "FT_response", "response"]


# --- create a new dataframe for lexical analysis ---
lex_df = pd.DataFrame()
lex_df["ID"] = df["ID"]


# --- compute word count per row for each column ---
for col in cols:
    lex_df[f"{col}_word_count"] = df[col].apply(spacy_word_count)


# --- save to CSV ---
lex_df.to_csv("/work/NLP2025/data/lexical_analysis.csv", index=False)

lex_df.head()

,ID,Context_word_count,Human_response_word_count,FT_response_word_count,response_word_count
0,context_1,123,172,291,130
1,context_2,102,223,257,115
2,context_3,223,235,371,172
3,context_4,86,587,257,77
4,context_5,39,255,168,76


In [13]:
# Average number of words used in each column

# Load your lexical analysis file
lex = pd.read_csv("/work/NLP2025/data/lexical_analysis.csv")

# Select only the word-count columns (everything except ID)
word_count_cols = [col for col in lex.columns if col.endswith("_word_count")]

# Compute and print the average word count for each column
for col in word_count_cols:
    avg = lex[col].mean()
    print(f"Average words in {col}: {avg:.2f}")

Average words in Context_word_count: 64.09
Average words in Human_response_word_count: 178.73
Average words in FT_response_word_count: 207.57
Average words in response_word_count: 91.25


In [15]:
for col in word_count_cols:
    col_min = lex[col].min()
    col_max = lex[col].max()
    print(f"{col}: min = {col_min}, max = {col_max}")


Context_word_count: min = 4, max = 528
Human_response_word_count: min = 9, max = 929
FT_response_word_count: min = 82, max = 685
response_word_count: min = 1, max = 180


In [17]:
# Load the lexical analysis output
lex = pd.read_csv("/work/NLP2025/data/lexical_analysis.csv")

# Select columns ending with _word_count
word_count_cols = [col for col in lex.columns if col.endswith("_word_count")]

# Compute summary statistics
summary_df = pd.DataFrame({
    "Mean": lex[word_count_cols].mean(),
    "Min": lex[word_count_cols].min(),
    "Max": lex[word_count_cols].max(),
    "StdDev": lex[word_count_cols].std()
})

# Round results for nicer presentation
summary_df = summary_df.round(2)

# Print the table
print(summary_df)

# Save to CSV
#summary_df.to_csv("word_count_summary.csv")

                             Mean  Min  Max  StdDev
Context_word_count          64.09    4  528   58.83
Human_response_word_count  178.73    9  929  114.94
FT_response_word_count     207.57   82  685   67.01
response_word_count         91.25    1  180   40.64


In [ ]:
# from Minas classsss
import re # THIS LINE IS NOT NEEDED IF YOU HAVE ALREADY IMPORTED 're'

def get_words(text:str) -> list[str]: # note: ':str' in params and 'list[str]' are called "type hints" and documents what types of variables should be input and output
    """
    This function takes 'text' as input and makes it lowercase, removes basic punctuation, and splits the text into a list of words.

    Args 
        text: string to be processed 
    
    Returns
        words: list of all words in 'text'
    """
    # pre-processing
    lowercased_text = text.lower()
    clean_text = re.sub(r"[.,!?:]", "", lowercased_text) 

    words = clean_text.split()

    return words

# applying it!
student_words = get_words(student_text)
chatgpt_words = get_words(chatgpt_text)

# only printing student words as we have already looked at chatgptwords
print(student_words)

In [18]:
# Total number of sentences and average sentence length
def sentence_count(text):
    """Return number of sentences detected by spaCy."""
    if not isinstance(text, str) or text.strip() == "":
        return 0
    doc = nlp(text)
    return len(list(doc.sents))


def avg_sentence_length(text):
    """Return avg words per sentence using spaCy alphabetic tokens."""
    if not isinstance(text, str) or text.strip() == "":
        return 0
    doc = nlp(text)
    sentences = list(doc.sents)
    if len(sentences) == 0:
        return 0
    
    # total words across sentences
    total_words = sum(len([t for t in sent if t.is_alpha]) for sent in sentences)
    
    return total_words / len(sentences)


In [19]:
cols = ["Context", "Human_response", "FT_response", "response"]

# Create new dataframe to store results
sentence_df = pd.DataFrame()
sentence_df["ID"] = df["ID"]

# Compute per-row metrics
for col in cols:
    sentence_df[f"{col}_sentence_count"] = df[col].apply(sentence_count)
    sentence_df[f"{col}_avg_sentence_length"] = df[col].apply(avg_sentence_length)


In [20]:
# Select the new columns
metrics = [col for col in sentence_df.columns if col != "ID"]

# Build summary table
summary = pd.DataFrame({
    "Mean": sentence_df[metrics].mean(),
    "Min": sentence_df[metrics].min(),
    "Max": sentence_df[metrics].max(),
    "StdDev": sentence_df[metrics].std()
}).round(2)

print(summary)


                                     Mean   Min    Max  StdDev
Context_sentence_count               5.13  1.00  34.00    4.06
Context_avg_sentence_length         12.50  4.00  40.00    4.04
Human_response_sentence_count        9.85  1.00  48.00    7.24
Human_response_avg_sentence_length  19.82  7.12  89.33    6.90
FT_response_sentence_count          13.03  5.00  43.00    4.57
FT_response_avg_sentence_length     16.52  8.81  41.40    4.06
response_sentence_count              6.50  1.00  21.00    4.46
response_avg_sentence_length        15.83  1.00  39.50    4.83


In [21]:
# Type-Token Ratio (TTR) → Unique words / total words

#Define function using spaCy
def ttr_spacy(text):
    """Compute Type–Token Ratio for a text using spaCy alphabetic tokens."""
    if not isinstance(text, str) or text.strip() == "":
        return 0
    
    doc = nlp(text.lower())  # lowercase so 'Hello' and 'hello' count as same type
    tokens = [t.text for t in doc if t.is_alpha]

    if len(tokens) == 0:
        return 0
    
    return len(set(tokens)) / len(tokens)


In [22]:
cols = ["Context", "Human_response", "FT_response", "response"]

ttr_df = pd.DataFrame()
ttr_df["ID"] = df["ID"]

for col in cols:
    ttr_df[f"{col}_TTR"] = df[col].apply(ttr_spacy)


In [23]:
ttr_metrics = [col for col in ttr_df.columns if col.endswith("_TTR")]

ttr_summary = pd.DataFrame({
    "Mean": ttr_df[ttr_metrics].mean(),
    "Min": ttr_df[ttr_metrics].min(),
    "Max": ttr_df[ttr_metrics].max(),
    "StdDev": ttr_df[ttr_metrics].std()
}).round(3)

print(ttr_summary)


                     Mean    Min    Max  StdDev
Context_TTR         0.753  0.398  1.000   0.115
Human_response_TTR  0.623  0.359  1.000   0.104
FT_response_TTR     0.750  0.447  0.924   0.070
response_TTR        0.773  0.563  1.000   0.071
